# Diagnóstico 04 — Estratégias de leitura e filtros Parquet

## Objetivo

Comparar três estratégias de leitura para as bases `grupo` e `dsp`:

- **A — Leitura completa:** lê as colunas projetadas de todo o Row Group, sem filtro territorial.
- **B — Filtro durante a leitura:** utiliza `pyarrow.dataset.Scanner` com expressão de filtro.
- **C — Filtro após a leitura:** lê o Row Group completo e aplica `pyarrow.compute` sobre a tabela Arrow.

A comparação mede RSS, USS, Memory Pool, pico de alocado, tempo, linhas e buffers da tabela final.

O teste procura responder: **aplicar o filtro no leitor reduz o pico de memória ou apenas reduz o tamanho da tabela resultante?**

Os cenários rodam em processos Python independentes. Não altera o dashboard nem os Parquets.

## 1. Configuração

Execute a partir da raiz de `app_render`. As colunas e caminhos reproduzem o `features_names.py` analisado anteriormente.

In [1]:
from pathlib import Path
import os, sys, json, subprocess, importlib
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dash_utils" / "config.py").exists():
    candidatos = [
        Path.cwd().parent,
        Path.cwd() / "app_render",
        Path.home() / "PycharmProjects" / "Projeto_ENEM" / "app_render",
    ]
    PROJECT_ROOT = next(
        (p.resolve() for p in candidatos if (p / "dash_utils" / "config.py").exists()),
        None
    )
if PROJECT_ROOT is None:
    raise FileNotFoundError("Não encontrei dash_utils/config.py. Ajuste PROJECT_ROOT.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

config = importlib.import_module("dash_utils.config")
CONSUMO_DIR = Path(config.CONSUMO_DIR)
UF = "PA"
REFERENCIA = "ESC"  # "ESC", "PROVA" ou None

UF_DIR = CONSUMO_DIR / f"uf={UF}"
ARQUIVOS = {
    "grupo": UF_DIR / "desempenho_grupos_municipal.parquet",
    "dsp": UF_DIR / "desempenho_sexo_faixa_etaria_municipal.parquet",
}
COLUNAS = {
    "grupo": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "tipo_grupo", "grupo", "disciplina", "media", "n_validos",
        "mediana", "std", "q1", "q3",
    ],
    "dsp": [
        "ano", "CO_MUNICIPIO_TERRITORIO", "TIPO_REFERENCIA_TERRITORIAL",
        "sexo", "faixa_etaria", "disciplina", "n_participantes",
        "n_validos", "media", "std",
    ],
}
for nome, caminho in ARQUIVOS.items():
    if not caminho.exists():
        raise FileNotFoundError(caminho)
    print(f"{nome}: {caminho} | {caminho.stat().st_size / 1024**2:.2f} MB")

print("Projeto:", PROJECT_ROOT)
print("Python:", sys.version)
print("PyArrow:", pa.__version__)
print("Memory Pool:", pa.default_memory_pool().backend_name)
print("UF:", UF, "| Referência:", REFERENCIA)

grupo: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_grupos_municipal.parquet | 8.57 MB
dsp: /home/akel/PycharmProjects/Projeto_ENEM/app_render/data/consumo/uf=PA/desempenho_sexo_faixa_etaria_municipal.parquet | 10.10 MB
Projeto: /home/akel/PycharmProjects/Projeto_ENEM/app_render
Python: 3.11.18 (main, Oct 26 2026, 11:35:17) [GCC 15.3.0]
PyArrow: 26.0.1
Memory Pool: mimalloc
UF: PA | Referência: ESC


## 2. Conferência da estrutura física

Confirma row groups e estatísticas da coluna usada no filtro. As estatísticas ajudam a entender se o leitor tem informação suficiente para descartar row groups inteiros. Com um único Row Group, o filtro pode reduzir linhas no resultado sem evitar a leitura das colunas necessárias daquele grupo.

In [2]:
linhas=[]
for base, caminho in ARQUIVOS.items():
    pf=pq.ParquetFile(caminho)
    meta=pf.metadata
    for i in range(meta.num_row_groups):
        rg=meta.row_group(i)
        idx=next(
            (j for j in range(rg.num_columns)
             if rg.column(j).path_in_schema=="TIPO_REFERENCIA_TERRITORIAL"),
            None
        )
        stat=None
        if idx is not None:
            st=rg.column(idx).statistics
            if st is not None:
                stat={
                    "min": str(st.min),
                    "max": str(st.max),
                    "distinct_count": st.distinct_count,
                    "null_count": st.null_count,
                }
        linhas.append({
            "base":base,
            "row_group":i,
            "linhas":rg.num_rows,
            "colunas":rg.num_columns,
            "total_comprimido_mb":rg.total_byte_size/1024**2,
            "estatisticas_referencia":stat,
        })
display(pd.DataFrame(linhas))

,base,row_group,linhas,colunas,total_comprimido_mb,estatisticas_referencia
0,grupo,0,337214,16,10.338845,"{'min': 'ESC', 'max': 'RESIDENCIA', 'distinct_..."
1,dsp,0,451205,17,12.507343,"{'min': 'ESC', 'max': 'PROVA', 'distinct_count..."


## 3. Worker: três estratégias em processos independentes

O worker registra fotografias de memória em cada etapa. Para o cenário B, utiliza `pyarrow.dataset` e filtro no `Scanner`. Para o cenário C, aplica `table.filter()` depois de materializar a leitura completa.

O filtro ESC considera `ESC` e `RESIDENCIA`, reproduzindo a regra usada no projeto. PROVA considera apenas `PROVA`.

In [3]:
WORKER = r'''
import sys, json, gc, time
import psutil
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import pyarrow.compute as pc

p=json.loads(sys.argv[1])
proc=psutil.Process()
pool=pa.default_memory_pool()

def snap():
    m=proc.memory_full_info()
    return {
        "rss_mb":m.rss/1024**2,
        "uss_mb":getattr(m,"uss",0)/1024**2,
        "pool_bytes_mb":pool.bytes_allocated()/1024**2,
        "pool_max_mb":pool.max_memory()/1024**2,
        "total_allocated_mb":pa.total_allocated_bytes()/1024**2,
    }

def filtro_arrow(ref):
    if ref=="ESC":
        vals=["ESC","RESIDENCIA"]
    elif ref=="PROVA":
        vals=["PROVA"]
    else:
        return None
    return ds.field("TIPO_REFERENCIA_TERRITORIAL").isin(vals)

out={
    "base":p["base"],
    "cenario":p["cenario"],
    "referencia":p["referencia"],
    "memory_map":p["memory_map"],
}
out["baseline"]=snap()
t0=time.perf_counter()

if p["cenario"] in ("A_completa","C_pos_leitura"):
    pf=pq.ParquetFile(p["arquivo"],memory_map=p["memory_map"])
    out["apos_abrir"]=snap()
    table=pf.read_row_group(0,columns=p["columns"])
    out["apos_materializar"]=snap()
    if p["cenario"]=="C_pos_leitura" and p["referencia"] is not None:
        if p["referencia"]=="ESC":
            vals=["ESC","RESIDENCIA"]
        else:
            vals=["PROVA"]
        arr=table["TIPO_REFERENCIA_TERRITORIAL"]
        mask=pc.is_in(arr,value_set=pa.array(vals,type=arr.type))
        table_final=table.filter(mask)
        out["apos_filtro"]=snap()
    else:
        table_final=table
elif p["cenario"]=="B_filtro_scanner":
    dataset=ds.dataset(p["arquivo"],format="parquet")
    scanner=dataset.scanner(
        columns=p["columns"],
        filter=filtro_arrow(p["referencia"]),
        use_threads=p["use_threads"],
    )
    out["apos_criar_scanner"]=snap()
    table_final=scanner.to_table()
    out["apos_scanner_to_table"]=snap()
else:
    raise ValueError("Cenário inválido")

t1=time.perf_counter()
out["tempo_total_s"]=t1-t0
out["linhas_finais"]=table_final.num_rows
out["colunas_finais"]=table_final.num_columns
out["table_nbytes_mb"]=table_final.nbytes/1024**2
out["table_buffers_mb"]=table_final.get_total_buffer_size()/1024**2
out["schema"]={f.name:str(f.type) for f in table_final.schema}

# Captura o estado antes de liberar a tabela.
out["antes_del"]=snap()
del table_final
if "table" in locals():
    del table
if "pf" in locals():
    del pf
if "scanner" in locals():
    del scanner
if "dataset" in locals():
    del dataset
gc.collect()
out["apos_del_gc"]=snap()

print(json.dumps(out))
'''

def executar(base, cenario, referencia=REFERENCIA, memory_map=False, use_threads=True):
    payload={
        "base":base,
        "cenario":cenario,
        "referencia":referencia,
        "arquivo":str(ARQUIVOS[base]),
        "columns":COLUNAS[base],
        "memory_map":memory_map,
        "use_threads":use_threads,
    }
    proc=subprocess.run(
        [sys.executable,"-c",WORKER,json.dumps(payload)],
        cwd=str(PROJECT_ROOT),capture_output=True,text=True
    )
    if proc.returncode:
        raise RuntimeError(
            f"Erro {base}/{cenario}:\\nSTDOUT:\\n{proc.stdout}\\nSTDERR:\\n{proc.stderr}"
        )
    linhas=[x for x in proc.stdout.splitlines() if x.strip().startswith("{")]
    if not linhas:
        raise RuntimeError(f"Worker sem JSON: {proc.stdout}")
    return json.loads(linhas[-1])

## 4. Execução principal — A, B e C

Executa os três cenários para `grupo` e `dsp`. A tabela final compara tempo, linhas, buffers, delta USS e pico do Memory Pool.

In [4]:
CENARIOS=["A_completa","B_filtro_scanner","C_pos_leitura"]
resultados=[]
for base in ["grupo","dsp"]:
    for cenario in CENARIOS:
        ref=None if cenario=="A_completa" else REFERENCIA
        r=executar(base,cenario,referencia=ref,memory_map=False)
        resultados.append(r)
        print(
            f"{base:6s} | {cenario:18s} | linhas={r['linhas_finais']:>8} | "
            f"USS final={r['antes_del']['uss_mb']:.1f} MB | "
            f"pool={r['antes_del']['pool_bytes_mb']:.1f} MB | "
            f"buffers={r['table_buffers_mb']:.1f} MB"
        )

def resumo(r):
    b=r["baseline"]
    a=r["antes_del"]
    d=r["apos_del_gc"]
    return {
        "base":r["base"],
        "referencia": r["referencia"], 
        "cenario":r["cenario"],
        "linhas_finais":r["linhas_finais"],
        "tempo_total_s":r["tempo_total_s"],
        "delta_uss_mb":a["uss_mb"]-b["uss_mb"],
        "delta_rss_mb":a["rss_mb"]-b["rss_mb"],
        "delta_pool_mb":a["pool_bytes_mb"]-b["pool_bytes_mb"],
        "pool_max_mb":a["pool_max_mb"],
        "table_buffers_mb":r["table_buffers_mb"],
        "uss_apos_del_mb":d["uss_mb"],
        "pool_apos_del_mb":d["pool_bytes_mb"],
    }

df_resultados=pd.DataFrame([resumo(r) for r in resultados])
display(df_resultados)

grupo  | A_completa         | linhas=  337214 | USS final=184.0 MB | pool=46.7 MB | buffers=40.1 MB
grupo  | B_filtro_scanner   | linhas=  205400 | USS final=216.3 MB | pool=24.4 MB | buffers=24.4 MB
grupo  | C_pos_leitura      | linhas=  205400 | USS final=206.1 MB | pool=71.1 MB | buffers=24.3 MB
dsp    | A_completa         | linhas=  451205 | USS final=178.0 MB | pool=50.9 MB | buffers=45.1 MB
dsp    | B_filtro_scanner   | linhas=  263761 | USS final=214.2 MB | pool=26.3 MB | buffers=26.3 MB
dsp    | C_pos_leitura      | linhas=  263761 | USS final=206.0 MB | pool=77.3 MB | buffers=26.2 MB


,base,referencia,cenario,linhas_finais,tempo_total_s,delta_uss_mb,delta_rss_mb,delta_pool_mb,pool_max_mb,table_buffers_mb,uss_apos_del_mb,pool_apos_del_mb
0,grupo,NaN,A_completa,337214,0.031665,126.714844,131.714844,46.725159,58.771057,40.133999,183.992188,0.000000
1,grupo,ESC,B_filtro_scanner,205400,0.036732,159.035156,165.667969,24.419495,71.175903,24.416330,216.347656,0.000000
2,grupo,ESC,C_pos_leitura,205400,0.044628,148.792969,154.468750,71.075623,71.859192,24.269329,206.078125,3.882935
3,dsp,NaN,A_completa,451205,0.033717,120.683594,125.687500,50.904419,63.732666,45.132332,177.972656,0.000000
4,dsp,ESC,B_filtro_scanner,263761,0.043019,156.996094,163.621094,26.347717,77.283386,26.344055,214.214844,0.000000
5,dsp,ESC,C_pos_leitura,263761,0.051672,148.746094,154.421875,77.262634,78.268860,26.249600,206.015625,5.198547


## 5. Comparação do filtro: ESC e PROVA

Repete os cenários B e C para ESC e PROVA. O objetivo é avaliar se a estratégia do filtro altera o pico de memória de acordo com a seletividade.

In [5]:
resultados_refs=[]
for base in ["grupo","dsp"]:
    for ref in ["ESC","PROVA"]:
        for cenario in ["B_filtro_scanner","C_pos_leitura"]:
            r=executar(base,cenario,referencia=ref,memory_map=False)
            resultados_refs.append(r)

df_refs=pd.DataFrame([resumo(r) for r in resultados_refs])
display(df_refs.sort_values(["base","referencia","cenario"]))

,base,referencia,cenario,linhas_finais,tempo_total_s,delta_uss_mb,delta_rss_mb,delta_pool_mb,pool_max_mb,table_buffers_mb,uss_apos_del_mb,pool_apos_del_mb
4,dsp,ESC,B_filtro_scanner,263761,0.043875,150.992188,157.644531,26.347717,77.283386,26.344055,208.292969,0.000000
5,dsp,ESC,C_pos_leitura,263761,0.052709,146.746094,152.421875,77.262634,78.268860,26.249600,204.078125,5.198547
6,dsp,PROVA,B_filtro_scanner,187444,0.039994,160.984375,167.628906,19.007202,69.942871,19.003725,218.226562,0.000000
7,dsp,PROVA,C_pos_leitura,187444,0.050534,144.750000,150.425781,69.949463,70.664551,18.936564,202.089844,5.198547
0,grupo,ESC,B_filtro_scanner,205400,0.037611,157.019531,163.714844,24.419495,71.175903,24.416330,214.300781,0.000000
1,grupo,ESC,C_pos_leitura,205400,0.045436,148.781250,154.460938,71.075623,71.859192,24.269329,205.992188,3.882935
2,grupo,PROVA,B_filtro_scanner,131814,0.038301,163.019531,169.652344,15.882141,62.638550,15.878679,220.273438,0.000000
3,grupo,PROVA,C_pos_leitura,131814,0.042626,140.781250,146.457031,62.590393,63.093262,15.784309,198.097656,3.882935


## 6. Scanner com threads ligadas e desligadas

Compara `use_threads=True` e `False` no cenário de filtro durante a leitura. Isso ajuda a verificar se a leitura paralela altera o pico de memória.

In [6]:
resultados_threads=[]
for base in ["grupo","dsp"]:
    for threads in [True,False]:
        r=executar(
            base,"B_filtro_scanner",
            referencia=REFERENCIA,
            memory_map=False,
            use_threads=threads,
        )
        resultados_threads.append({
            "base":base,
            "use_threads":threads,
            "linhas":r["linhas_finais"],
            "tempo_s":r["tempo_total_s"],
            "delta_uss_mb":r["antes_del"]["uss_mb"]-r["baseline"]["uss_mb"],
            "pool_max_mb":r["antes_del"]["pool_max_mb"],
            "buffers_mb":r["table_buffers_mb"],
        })
display(pd.DataFrame(resultados_threads))

,base,use_threads,linhas,tempo_s,delta_uss_mb,pool_max_mb,buffers_mb
0,grupo,True,205400,0.039992,169.011719,71.207153,24.416330
1,grupo,False,205400,0.057940,128.457031,71.162598,24.416330
2,dsp,True,263761,0.042243,156.996094,77.314636,26.344055
3,dsp,False,263761,0.066567,126.457031,77.266052,26.344055


## 7. Conferência dos resultados e interpretação

### O que observar

- **B reduz o delta USS e o pico do pool em relação a C:** o filtro durante a leitura evita parte das alocações temporárias.
- **B e C têm consumo parecido:** com um único Row Group, o leitor provavelmente precisa materializar grande parte das colunas antes de aplicar o predicado.
- **B produz menos linhas/buffers, mas o pico é semelhante:** o filtro reduz o resultado final, mas não reduz significativamente o custo de leitura.
- **A é muito maior que B:** o filtro tem benefício de memória quando aplicado no scanner.
- **USS aumenta muito mais que o pool:** parte relevante das alocações ocorre fora do Memory Pool padrão ou é mantida pelo alocador.
- **`pool_max` é muito maior que `pool_bytes`:** houve alocação temporária durante a execução.

### Limites importantes

1. `pyarrow.dataset.Scanner` aplica o predicado durante o scan, mas isso não garante que o Parquet consiga descartar páginas ou row groups. Depende de estatísticas e organização física.
2. O arquivo tem um Row Group, conforme inventário anterior. Nesse cenário, o leitor pode precisar descomprimir/inspecionar grande parte das colunas para filtrar as linhas.
3. O cenário C usa `table.filter()` após a leitura e pode manter temporariamente a tabela original e a filtrada simultaneamente.
4. `pool_bytes` e `pool_max` cobrem o Memory Pool padrão, não necessariamente todas as alocações nativas.
5. Compare principalmente cenários em processos independentes e com o mesmo filtro.

### Decisão após os resultados

Se B reduzir significativamente o pico, vale testar o scanner como estratégia de leitura para as bases maiores.

Se B e C forem semelhantes, o próximo caminho é investigar a organização física dos Parquets (row groups, ordenação/particionamento e estatísticas), não apenas trocar a API de leitura.

**Não altere `read_allfiles2()` ainda.**